In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG crypto_pipeline")

DataFrame[]

In [0]:
# Databricks notebook source

# ==========================================================
# GOLD LAYER
#
# Notebook:
# 03_gold_top_movers
#
# Purpose:
# Daily ranking of cryptocurrencies based on
# 24-hour price change percentage.
#
# Grain:
# One row per Coin per Day
#
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "crypto_pipeline"

FACT_TABLE = f"{CATALOG}.silver.fact_coin_price"
GOLD_TABLE = f"{CATALOG}.gold.top_movers"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# Create Gold Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {GOLD_TABLE}

(

coin_sk STRING,

coin_id STRING,

observation_date DATE,

price_change_percentage_24h DOUBLE,

rank_position INT,

movement STRING

)

USING DELTA

""")

# ==========================================================
# Read Fact Table
# ==========================================================

fact_df = spark.table(FACT_TABLE)

fact_df = fact_df.withColumn(

    "observation_date",

    F.to_date("observation_ts")

)

display(fact_df)

coin_sk,coin_id,observation_ts,current_price,market_cap,total_volume,market_cap_rank,price_change_24h,price_change_percentage_24h,observation_date
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,2026-07-22T08:58:57.519Z,0.622088,1.670620127E9,6.3981992E7,46,-0.00839127419145902,-1.33094,2026-07-22
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,2026-07-22T08:59:21.134Z,0.622088,1.670620127E9,6.3981992E7,46,-0.00839127419145902,-1.33094,2026-07-22
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:58:57.519Z,6.51,2.812036807E9,1.21168985E8,32,-0.14037606851970175,-2.10982,2026-07-22
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:59:21.134Z,6.51,2.812036807E9,1.21168985E8,32,-0.14037606851970175,-2.10982,2026-07-22
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:58:57.519Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542,2026-07-22
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:59:21.134Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542,2026-07-22
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:58:57.519Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796,2026-07-22
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:59:21.134Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796,2026-07-22
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,2026-07-22T08:58:57.519Z,220.78,4.430395854E9,8.4996954E7,23,-2.645743369264551,-1.1842,2026-07-22
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,2026-07-22T08:59:21.134Z,220.78,4.430395854E9,8.4996954E7,23,-2.645743369264551,-1.1842,2026-07-22


In [0]:
ranking_window = (

    Window

    .partitionBy("observation_date")

    .orderBy(

        F.desc("price_change_percentage_24h")

    )

)

top_movers = (

    fact_df

    .withColumn(

        "rank_position",

        F.row_number().over(ranking_window)

    )

)

In [0]:
top_movers = (

    top_movers

    .withColumn(

        "movement",

        F.when(

            F.col("price_change_percentage_24h") > 0,

            "Gainer"

        )

        .when(

            F.col("price_change_percentage_24h") < 0,

            "Loser"

        )

        .otherwise(

            "Neutral"

        )

    )

)

In [0]:
from pyspark.sql.window import Window

latest_window = (
    Window
    .partitionBy("coin_sk", "observation_date")
    .orderBy(F.col("observation_ts").desc())
)

top_movers_stage = (
    top_movers
    .withColumn(
        "rn",
        F.row_number().over(latest_window)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        "coin_sk",
        "coin_id",
        "observation_date",
        "price_change_percentage_24h",
        "rank_position",
        "movement"
    )
)
display(top_movers_stage)

print(

    "Rows:",

    top_movers_stage.count()

)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7718852247748516>, line 6
      1 from pyspark.sql.window import Window
      3 latest_window = (
      4     Window
      5     .partitionBy("coin_sk", "observation_date")
----> 6     .orderBy(F.col("observation_ts").desc())
      7 )
      9 top_movers_stage = (
     10     top_movers
     11     .withColumn(
   (...)
     24     )
     25 )
     26 display(top_movers_stage)

NameError: name 'F' is not defined

In [0]:
from delta.tables import DeltaTable

gold_delta = DeltaTable.forName(

    spark,

    GOLD_TABLE

)

(

    gold_delta.alias("target")

    .merge(

        top_movers_stage.alias("source"),

        """

        target.coin_sk = source.coin_sk

        AND

        target.observation_date = source.observation_date

        """

    )

    .whenMatchedUpdate(

        set={

            "price_change_percentage_24h":

                "source.price_change_percentage_24h",

            "rank_position":

                "source.rank_position",

            "movement":

                "source.movement"

        }

    )

    .whenNotMatchedInsert(

        values={

            "coin_sk":

                "source.coin_sk",

            "coin_id":

                "source.coin_id",

            "observation_date":

                "source.observation_date",

            "price_change_percentage_24h":

                "source.price_change_percentage_24h",

            "rank_position":

                "source.rank_position",

            "movement":

                "source.movement"

        }

    )

    .execute()

)

print("Top Movers Merge Completed.")

Top Movers Merge Completed.


In [0]:
gold_df = spark.table(GOLD_TABLE)

display(

    gold_df.orderBy(

        "observation_date",

        "rank_position"

    )

)

print(

    "Total Rows:",

    gold_df.count()

)

coin_sk,coin_id,observation_date,price_change_percentage_24h,rank_position,movement
8c941d974538859fe31b9c1cff2356c1995838142ba8fa14310f557bb2546daa,hedera-hashgraph,2026-07-22,5.12698,1,Gainer
8c941d974538859fe31b9c1cff2356c1995838142ba8fa14310f557bb2546daa,hedera-hashgraph,2026-07-22,5.12698,2,Gainer
3b65a170fd4ddea6f2b105aa6ecb82aeb0cf642fc3ccb1358392ae6be7e1def0,rain,2026-07-22,5.03863,3,Gainer
3b65a170fd4ddea6f2b105aa6ecb82aeb0cf642fc3ccb1358392ae6be7e1def0,rain,2026-07-22,5.03863,4,Gainer
e2f96ad9165015037c46eab0b09fa5efb5a4ac4777053f7fa8869290b479db80,ondo-finance,2026-07-22,3.09239,5,Gainer
e2f96ad9165015037c46eab0b09fa5efb5a4ac4777053f7fa8869290b479db80,ondo-finance,2026-07-22,3.09239,6,Gainer
c00314f175383d608c77296a3493d91cdb2c1ec381d1e8f345ce9c447064d2e1,the-open-network,2026-07-22,2.59959,7,Gainer
c00314f175383d608c77296a3493d91cdb2c1ec381d1e8f345ce9c447064d2e1,the-open-network,2026-07-22,2.59959,8,Gainer
314b8687941b8ffc3e7752d655b21546c41cfdcba9d04feed56d0b34852217d2,monero,2026-07-22,1.82588,9,Gainer
314b8687941b8ffc3e7752d655b21546c41cfdcba9d04feed56d0b34852217d2,monero,2026-07-22,1.82588,10,Gainer


Total Rows: 100


In [0]:
display(

    gold_df

    .filter(

        F.col("movement") == "Gainer"

    )

    .orderBy(

        "observation_date",

        "rank_position"

    )

    .limit(10)

)

coin_sk,coin_id,observation_date,price_change_percentage_24h,rank_position,movement
8c941d974538859fe31b9c1cff2356c1995838142ba8fa14310f557bb2546daa,hedera-hashgraph,2026-07-22,5.12698,1,Gainer
8c941d974538859fe31b9c1cff2356c1995838142ba8fa14310f557bb2546daa,hedera-hashgraph,2026-07-22,5.12698,2,Gainer
3b65a170fd4ddea6f2b105aa6ecb82aeb0cf642fc3ccb1358392ae6be7e1def0,rain,2026-07-22,5.03863,3,Gainer
3b65a170fd4ddea6f2b105aa6ecb82aeb0cf642fc3ccb1358392ae6be7e1def0,rain,2026-07-22,5.03863,4,Gainer
e2f96ad9165015037c46eab0b09fa5efb5a4ac4777053f7fa8869290b479db80,ondo-finance,2026-07-22,3.09239,5,Gainer
e2f96ad9165015037c46eab0b09fa5efb5a4ac4777053f7fa8869290b479db80,ondo-finance,2026-07-22,3.09239,6,Gainer
c00314f175383d608c77296a3493d91cdb2c1ec381d1e8f345ce9c447064d2e1,the-open-network,2026-07-22,2.59959,7,Gainer
c00314f175383d608c77296a3493d91cdb2c1ec381d1e8f345ce9c447064d2e1,the-open-network,2026-07-22,2.59959,8,Gainer
314b8687941b8ffc3e7752d655b21546c41cfdcba9d04feed56d0b34852217d2,monero,2026-07-22,1.82588,9,Gainer
314b8687941b8ffc3e7752d655b21546c41cfdcba9d04feed56d0b34852217d2,monero,2026-07-22,1.82588,10,Gainer


In [0]:
display(

    gold_df

    .filter(

        F.col("movement") == "Loser"

    )

    .orderBy(

        F.desc("rank_position")

    )

    .limit(10)

)

coin_sk,coin_id,observation_date,price_change_percentage_24h,rank_position,movement
8ff3179a3e298e8b1a9a01812be6cfc086087b1adf0355bcbd98fc4432edea36,hyperliquid,2026-07-22,-6.66412,100,Loser
8ff3179a3e298e8b1a9a01812be6cfc086087b1adf0355bcbd98fc4432edea36,hyperliquid,2026-07-22,-6.66412,99,Loser
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,near,2026-07-22,-6.24059,98,Loser
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,near,2026-07-22,-6.24059,97,Loser
4a58a2c0e14acb98c656a990067f07e775e933535be0563e61def03cb586fa96,zcash,2026-07-22,-4.78993,96,Loser
4a58a2c0e14acb98c656a990067f07e775e933535be0563e61def03cb586fa96,zcash,2026-07-22,-4.78993,95,Loser
7aee56260d91a83514535669eb91b1d90b5bf21904882ed36544ba98ce361984,litecoin,2026-07-22,-2.62785,94,Loser
7aee56260d91a83514535669eb91b1d90b5bf21904882ed36544ba98ce361984,litecoin,2026-07-22,-2.62785,93,Loser
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22,-2.10982,92,Loser
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22,-2.10982,91,Loser
